# Backtesting with AlgoSystem

AlgoSystem takes an **equity curve** — a series of portfolio values that already happened — and turns it into performance metrics, a persisted run, and a quantstats tearsheet.

It does not generate signals or simulate order fills. You bring the equity curve; AlgoSystem measures it.

This notebook covers:

1. Running a backtest from a `pd.Series`
2. Reading the metrics
3. Comparing against a benchmark
4. Producing a tearsheet
5. Saving and reloading a run

Everything below uses synthetic data, so the notebook runs offline with no database and no network.

In [ ]:
import numpy as np
import pandas as pd

from algosystem import AlgoSystem

algo = AlgoSystem()

## 1. Build an equity curve

A daily series indexed by date. `freq="B"` gives business days, matching the 252-day annualisation factor.

In [ ]:
rng = np.random.default_rng(4)

index = pd.date_range("2021-01-01", periods=600, freq="B")
daily_returns = rng.normal(0.0004, 0.01, len(index))
equity = pd.Series(100_000 * np.cumprod(1 + daily_returns), index=index, name="strategy")

equity.head()

## 2. Run the backtest

`backtest()` returns a `BacktestResult` holding the equity curve, the computed metrics, and a run id.

In [ ]:
result = algo.backtest(equity)

# initial_capital and final_capital are Money value objects;
# total_return is a Percent. Use .amount and .as_percent to format them.
print(f"initial capital : {result.initial_capital.amount:,.0f}")
print(f"final capital   : {result.final_capital.amount:,.0f}")
print(f"total return    : {result.total_return.as_percent:.2f}%")
print(f"date range      : {result.date_range}")

## 3. Read the metrics

`result.metrics` is a `PerformanceMetrics` object. Names come from the `MetricKey` enum, so they are declared in exactly one place rather than restated per module.

A metric that could not be computed is `None` — never a fabricated substitute.

In [ ]:
metrics = result.metrics

for name in [
    "sharpe_ratio",
    "sortino_ratio",
    "annualized_return",
    "annualized_volatility",
    "max_drawdown",
    "calmar_ratio",
]:
    value = metrics.get(name)
    print(f"{name:<24} {value if value is None else round(value, 4)}")

In [ ]:
# The full set, as a plain dict. PerformanceMetrics has no .keys() —
# to_dict() is the accessor.
all_metrics = metrics.to_dict()
print(f"{len(all_metrics)} metrics computed\n")

pd.Series(all_metrics)

## 4. Compare against a benchmark

Passing `benchmark=` adds the relative metrics: alpha, beta, correlation, and up/down capture ratios.

In [ ]:
bench_returns = rng.normal(0.0003, 0.009, len(index))
benchmark = pd.Series(100_000 * np.cumprod(1 + bench_returns), index=index, name="benchmark")

relative = algo.backtest(equity, benchmark=benchmark)

for name in ["alpha", "beta", "correlation", "capture_ratio_up", "capture_ratio_down"]:
    value = relative.metrics.get(name)
    print(f"{name:<20} {value if value is None else round(value, 4)}")

Because both series here are independent noise, the correlation and beta are near zero and the alpha is an artefact of the drift difference. On real data these are the numbers that say whether the strategy added anything the benchmark did not.

## 5. Tearsheet

Reporting is quantstats' tearsheet rather than a bespoke dashboard. It writes a self-contained HTML file — roughly 700 KB — and never opens a browser on its own.

In [ ]:
path = algo.tearsheet(relative, output="backtest_tearsheet.html")
print("written to:", path)

# To view it inline:
# from IPython.display import IFrame
# IFrame(str(path), width="100%", height=600)

## 6. Persisting a run

`save()`, `load()` and `compare()` need a repository. `AlgoSystem()` is constructed without one, so ask for it explicitly — a bare `algo.save(...)` raises `ConfigurationError`.

The in-memory repository below lasts for the life of the process. Swap in the Postgres repository to persist across sessions; the calling code does not change.

The repository assigns the run id on write, so saving two backtests never collides.

In [ ]:
from algosystem.backtesting.infrastructure.persistence.in_memory_repository import (
    InMemoryBacktestRunRepository,
)

store = AlgoSystem(repository=InMemoryBacktestRunRepository())

run_id = store.save(relative, name="demo-run")
print("saved as:", run_id)

reloaded = store.load(run_id)
print("reloaded sharpe:", round(reloaded.metrics.get("sharpe_ratio"), 4))

## What this does not tell you

Every metric above describes **one** equity curve. None of it says whether the strategy that produced that curve was selected by trying many parameter combinations and keeping the best — which is how a backtest ends up looking good by accident.

A Sharpe of 0.83 from a single a-priori strategy and a Sharpe of 0.83 that is the maximum over 80 configurations are very different claims, and the metrics here cannot distinguish them.

That is what `02_parameter_sensitivity.ipynb` addresses.